In [1]:
from pathlib import Path
import numpy as np, pickle
import libstempo as T
from enterprise.pulsar import Pulsar

# ---------- Paths ----------
BASE = Path("/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2")
PAR_DIR = BASE / "par"
TIM_DIR = BASE / "tim"
OUT_DIR = Path("/scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/sim_dataset2")
OUT_DIR.mkdir(exist_ok=True)
PKL_FILE = OUT_DIR / "G2D2_sim_injected.pkl"


In [2]:
# ---------- CW parameters ----------
CW = dict(
    fgw = 3.7e-9,
    log10_h = -13.668773493298787,
    gw_phi = 3.3335788713091694,
    gw_theta = 0.6387905062299246,
    inclination = 0.8412486994612669,
    phase0 = 0.24434609527920614,
    psi = 1.1187560505283651,
)
# Derived
h0 = 10**CW["log10_h"]
g_ra, g_dec = CW["gw_phi"], np.pi/2 - CW["gw_theta"]


In [3]:
# ---------- Geometry helper ----------
def antenna_factors(p_ra, p_dec, g_ra, g_dec, psi):
    ph = np.array([np.cos(p_dec)*np.cos(p_ra),
                   np.cos(p_dec)*np.sin(p_ra),
                   np.sin(p_dec)])
    kh = np.array([np.cos(g_dec)*np.cos(g_ra),
                   np.cos(g_dec)*np.sin(g_ra),
                   np.sin(g_dec)])
    x = np.array([-np.sin(g_ra), np.cos(g_ra), 0.0])
    y = np.cross(kh, x)
    c2, s2 = np.cos(2*psi), np.sin(2*psi)
    eplus  = np.outer(x,x) - np.outer(y,y)
    ecross = np.outer(x,y) + np.outer(y,x)
    eplus, ecross = c2*eplus+s2*ecross, -s2*eplus+c2*ecross
    denom = 1.0 - ph.dot(kh)
    Fp = 0.5*(ph @ eplus @ ph)/denom
    Fx = 0.5*(ph @ ecross @ ph)/denom
    return Fp, Fx, denom


# ---------- NEW FUNCTION: handles RAJ/DECJ *and* ELONG/ELAT ----------
def get_radec(psr):
    """Return RA [rad], DEC [rad] regardless of whether par uses RAJ/DECJ or ELONG/ELAT."""
    pars_upper = [p.upper() for p in psr.pars()]
    # Ecliptic case
    if "ELONG" in pars_upper and "ELAT" in pars_upper:
        elon = float(psr["ELONG"].val)
        elat = float(psr["ELAT"].val)
        # Convert to equatorial radians
        import astropy.units as u
        from astropy.coordinates import SkyCoord
        coord = SkyCoord(elon*u.rad, elat*u.rad, frame="barycentrictrueecliptic")
        pra, pdec = coord.icrs.ra.rad, coord.icrs.dec.rad
        return pra, pdec

    # Equatorial case
    for racand, deccand in [("RAJ","DECJ"),("RA","DEC"),("RA_","DEC_")]:
        if racand in pars_upper and deccand in pars_upper:
            pra = float(psr[psr.pars()[pars_upper.index(racand)]].val)
            pdec = float(psr[psr.pars()[pars_upper.index(deccand)]].val)
            return pra, pdec

    raise KeyError(f"No RA/DEC or ELONG/ELAT found for {psr.name}")


# ---------- Residuals using get_radec ----------
def cw_residuals(psr):
    fgw, iota, psi, phi0 = CW["fgw"], CW["inclination"], CW["psi"], CW["phase0"]
    pra, pdec = get_radec(psr)
    Fp, Fx, denom = antenna_factors(pra, pdec, g_ra, g_dec, psi)

    # libstempo stores TOAs as MJD; convert to seconds
    toas = psr.stoas.copy().astype(float) * 86400.0
    t_ref = toas.mean()

    cth = np.cos(iota)
    phiE = 2*np.pi*fgw*(toas - t_ref) + phi0
    splus  = 0.5*(1 + cth**2) * np.cos(phiE)
    scross = cth * np.sin(phiE)
    res = (h0 / (2*np.pi*fgw)) * (Fp*splus + Fx*scross)
    return res



In [4]:
# ---------- Inject and verify ----------
for par in sorted(PAR_DIR.glob("*.par")):
    name = par.stem
    tim = TIM_DIR / f"{name}.tim"
    psr = T.tempopulsar(str(par), str(tim))
    res = cw_residuals(psr)
    psr.stoas[:] += res/ 86400.0
    outtim = OUT_DIR / f"{name}_CW.tim"
    psr.savetim(str(outtim))
    # quick verification
    print(f"{name}: injected RMS = {np.std(res)*1e6:.2f} µs, saved -> {outtim.name}")

'''
# ---- Diagnostics ----
    print(name, "Fp Fx mean", np.mean(res))
    print("min=", res.min(), "max=", res.max())
    print("Fp, Fx check:")
    pra, pdec = get_radec(psr)
    Fp, Fx, denom = antenna_factors(pra, pdec, g_ra, g_dec, CW["psi"])
    print(Fp, Fx)
    print()

    psr.stoas[:] += res
    psr.savetim(str(OUT_DIR / f"{name}_CW.tim"))
'''

J0030+0451: injected RMS = 0.10 µs, saved -> J0030+0451_CW.tim
J0034-0534: injected RMS = 0.07 µs, saved -> J0034-0534_CW.tim
J0218+4232: injected RMS = 0.26 µs, saved -> J0218+4232_CW.tim
J0437-4715: injected RMS = 0.04 µs, saved -> J0437-4715_CW.tim
J0613-0200: injected RMS = 0.19 µs, saved -> J0613-0200_CW.tim
J0621+1002: injected RMS = 0.24 µs, saved -> J0621+1002_CW.tim
J0711-6830: injected RMS = 0.06 µs, saved -> J0711-6830_CW.tim
J0751+1807: injected RMS = 0.31 µs, saved -> J0751+1807_CW.tim
J0900-3144: injected RMS = 0.20 µs, saved -> J0900-3144_CW.tim
J1012+5307: injected RMS = 0.42 µs, saved -> J1012+5307_CW.tim
J1022+1001: injected RMS = 0.37 µs, saved -> J1022+1001_CW.tim
J1024-0719: injected RMS = 0.31 µs, saved -> J1024-0719_CW.tim
J1045-4509: injected RMS = 0.18 µs, saved -> J1045-4509_CW.tim
J1455-3330: injected RMS = 0.22 µs, saved -> J1455-3330_CW.tim
J1600-3053: injected RMS = 0.21 µs, saved -> J1600-3053_CW.tim
J1603-7202: injected RMS = 0.08 µs, saved -> J1603-7202

'\n# ---- Diagnostics ----\n    print(name, "Fp Fx mean", np.mean(res))\n    print("min=", res.min(), "max=", res.max())\n    print("Fp, Fx check:")\n    pra, pdec = get_radec(psr)\n    Fp, Fx, denom = antenna_factors(pra, pdec, g_ra, g_dec, CW["psi"])\n    print(Fp, Fx)\n    print()\n\n    psr.stoas[:] += res\n    psr.savetim(str(OUT_DIR / f"{name}_CW.tim"))\n'

In [5]:
# ---------- Build Enterprise-compatible PKL ----------
pulsars = []
for par in sorted(PAR_DIR.glob("*.par")):
    new_tim = OUT_DIR / f"{par.stem}_CW.tim"
    if new_tim.exists():
        psr_obj = Pulsar(str(par), str(new_tim), timing_package="tempo2")
        pulsars.append(psr_obj)
        print(f"✔ {par.stem}: {len(psr_obj.toas)} TOAs")

with open(PKL_FILE, "wb") as f:
    pickle.dump(pulsars, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n✅ Saved {len(pulsars)} injected pulsars to {PKL_FILE}")


✔ J0030+0451: 183 TOAs
✔ J0034-0534: 183 TOAs


✔ J0218+4232: 183 TOAs


✔ J0437-4715: 183 TOAs
✔ J0613-0200: 183 TOAs


✔ J0621+1002: 183 TOAs
✔ J0711-6830: 183 TOAs


✔ J0751+1807: 183 TOAs
✔ J0900-3144: 183 TOAs
✔ J1012+5307: 183 TOAs
✔ J1022+1001: 183 TOAs
✔ J1024-0719: 183 TOAs
✔ J1045-4509: 183 TOAs
✔ J1455-3330: 183 TOAs
✔ J1600-3053: 183 TOAs
✔ J1603-7202: 183 TOAs


✔ J1614-2230: 183 TOAs
✔ J1640+2224: 183 TOAs
✔ J1643-1224: 183 TOAs
✔ J1713+0747: 183 TOAs
✔ J1730-2304: 183 TOAs
✔ J1744-1134: 183 TOAs
✔ J1857+0943: 183 TOAs
✔ J1909-3744: 183 TOAs
✔ J1918-0642: 183 TOAs
✔ J1939+2134: 183 TOAs


✔ J1944+0907: 183 TOAs
✔ J2010-1323: 183 TOAs
✔ J2124-3358: 183 TOAs
✔ J2129-5721: 183 TOAs
✔ J2145-0750: 183 TOAs


✔ J2229+2643: 183 TOAs
✔ J2317+1439: 183 TOAs

✅ Saved 33 injected pulsars to /scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/sim_dataset2/G2D2_sim_injected.pkl


In [6]:
import libstempo as T, numpy as np
from pathlib import Path

PAR = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/par/J1713+0747.par"
TIM0 = "/scratch/na00078/projects/IPTA_MDC2/mdc2/group2/dataset_2/tim/J1713+0747.tim"          # original
TIM1 = "/scratch/na00078/projects/IPTA_MDC2/IPTA_MDC2_data/sim_dataset2/J1713+0747_CW.tim"  # injected

p0 = T.tempopulsar(PAR, TIM0)
p1 = T.tempopulsar(PAR, TIM1)

# residual added by injection = TOA shift (seconds)
r_injected = (p1.stoas - p0.stoas) * 86400.0
print("Injected residual RMS [µs]:", np.std(r_injected)*1e6)

# sanity: libstempo’s post-fit residuals for each file
print("Orig post-fit RMS [µs]:", np.std(p0.residuals())*1e6)
print("Inj  post-fit RMS [µs]:", np.std(p1.residuals())*1e6)


Injected residual RMS [µs]: 25188.32894874999234
Orig post-fit RMS [µs]: 8.931678958991763975
Inj  post-fit RMS [µs]: 1329.8859776327062837


In [7]:
print("Mean stoas diff (days):", np.mean(p1.stoas - p0.stoas))
print("Std stoas diff (days):", np.std(p1.stoas - p0.stoas))
print("Std stoas diff (µs):", np.std(p1.stoas - p0.stoas)*86400*1e6)


Mean stoas diff (days): 5.486901675451873252e-08
Std stoas diff (days): 2.915315850549767632e-07
Std stoas diff (µs): 25188.32894874999234
